In [1]:
import torch 
import lightning as L
import yaml
import sys, os
import pickle
sys.path.append('../')
# from lightning_scripts.lightning_classifier import LitWordAudioSetModel 
from lightning_scripts.lightning_ssl import LitAudioSSL 
from lightning_scripts.lightning_classifier_matched_speech_in_noise import LitWordAudioSetModel as LitWordAudioSetModelMatched
from lightning_scripts.jsinV3DataLoader_precombined_batched import jsinV3_precombined_all_signals
from torchmetrics.classification import Accuracy, BinaryPrecision
import numpy as np 
from tqdm import tqdm
from pathlib import Path
import robustness.audio_functions.audio_transforms as at

In [2]:


# get config and init trained model 
config_path = Path("model_configs/word_speaker_audioset_resnet18_MatchedDataset_shuffle_one_gpu.yaml")
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)
# print(config)

config['num_workers'] = 2
config['hparas']['batch_size'] = 124 # set to single-gpu size 

# update val set to use entire range 
# config['data']['eval_max'] = -1



checkpoint_dir = Path("model_checkpoints") / f"{config_path.stem}/checkpoints"
ckpt_paths = sorted(checkpoint_dir.glob("*.ckpt"), key=os.path.getctime)
ckpt_path = ckpt_paths[-1] # get latest checkpoint 
print(ckpt_path)

model_checkpoints/word_speaker_audioset_resnet18_MatchedDataset_shuffle_one_gpu/checkpoints/epoch=45-step=110354-best_train.ckpt


In [3]:
config['model']

{'arch_name': 'resnet_multi_task18',
 'arch_params': {'num_classes': {'signal/word_int': 794,
   'signal/speaker_int': 433,
   'noise/labels_int': 517},
  'pretrained': False}}

In [4]:
model = LitWordAudioSetModelMatched.load_from_checkpoint(checkpoint_path=ckpt_path, config=config)
model = torch.compile(model)
model = model.eval().cuda()


In [5]:
transforms = at.AudioCompose([
                    at.AudioToTensor(),
                    at.DBSPLNormalizeForegroundAndBackground(60),
                    at.UnsqueezeAudio(dim=0) # dim=0 here so batches of audio from dataloader will be (Batch, 1, Time)
                ])

In [6]:
def collate_fn(batch):
    batch = batch[0] # unbox wrapper added by dataloader 
    speech_batch = []
    noise_batch = []
    for (speech, noise) in zip(*batch[:2]):
        speech = transforms(speech, None)[0]
        noise = transforms(noise, None)[0]
        if speech is None:
            speech = torch.zeros(1,40_000)
        if noise is None:
            noise = torch.zeros(1,40_000)
        speech_batch.append(speech)
        noise_batch.append(noise)

    speech_batch = torch.cat(speech_batch).unsqueeze(1)
    noise_batch = torch.cat(noise_batch).unsqueeze(1)
    
    labels = batch[-1] # labels already collated 
    # convert labels to torch tensors 
    if isinstance(labels, dict):
        for task_key, task_labels in labels.items():
            labels[task_key] = torch.from_numpy(task_labels)
    else:
        labels = torch.from_numpy(labels) 
    # convert signal and noise into signal
    return speech_batch, noise_batch, labels 

In [7]:
val_dataset = jsinV3_precombined_all_signals(root="/mnt/ceph/users/jfeather/data/training_datasets_audio/JSIN_all_v3/subsets/",
                                                 train=False,
                                                 transform=None,
                                                 batch_size=config['hparas']['batch_size'],
                                                 eval_max=5)
dataloader = torch.utils.data.DataLoader(
                                        val_dataset,
                                        batch_size=1,
                                        num_workers=config['num_workers'],
                                        shuffle=False,
                                        pin_memory=True,
                                        collate_fn=collate_fn
                                            )

In [8]:
len(val_dataset) * 96 

64800

In [25]:
batch = next(iter(dataloader))

In [26]:
speech, noise, labels = batch

In [27]:
noise.shape

torch.Size([96, 1, 40000])

In [28]:
labels.keys()

dict_keys(['signal/word_int', 'signal/speaker_int', 'noise/labels_binary_via_int'])

In [10]:
model_word_key = [key for key in config['data']['target_keys'] if 'word' in key][0]
model_speaker_key = [key for key in config['data']['target_keys'] if 'speaker' in key][0]
model_noise_key = [key for key in config['data']['target_keys'] if 'noise' in key][0]

In [11]:
model_noise_key



'noise/labels_int'

In [16]:
prec = BinaryPrecision()

word_acc = []
speaker_acc = []
noise_prec = []
with torch.no_grad():
    for ix, batch in enumerate(tqdm(dataloader)):
        speech, noise, labels = batch
        ### Get word and speaker outs
        speech_logits = model(speech.cuda())

        word_preds = speech_logits[model_word_key].softmax(-1).argmax(-1).cpu()
        speaker_preds = speech_logits[model_speaker_key].softmax(-1).argmax(-1).cpu()

        word_acc.append((word_preds == labels["signal/word_int"]).numpy().mean()) 
        speaker_acc.append((speaker_preds == labels["signal/speaker_int"]).numpy().mean()) 

        ### Get noise label outs 
        noise_logits = model(noise.cuda())[model_noise_key].cpu()
        noise_prec.append(prec(noise_logits, labels['noise/labels_binary_via_int']).item())



  0%|          | 0/675 [00:00<?, ?it/s]

100%|██████████| 675/675 [21:13<00:00,  1.89s/it]


In [17]:
np.mean(word_acc)

np.float64(0.8067502986857825)

In [18]:
np.mean(speaker_acc)

np.float64(0.6465352449223416)

In [19]:
np.mean(noise_prec)

np.float64(0.5255848915488631)